In [1]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"


# 1. Read data

In [2]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/splited/"

In [3]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = get_available_alleles(DATA_PATH)

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

ALLELES

['HLA-A*02:01']

# 2. Train HMMS

In [4]:
from hmm_logic_methods import train_model, save_model, add_more_states_and_reset_transitions, create_model_with_separated_amino_acids


NUM_RUNS = 2
TARGET_LENGTH = 9
TARGET_LENGTHS= [9]

## 2.1 Prepare model


In [5]:
per_allele_per_run_per_split_prepared_models = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_prepared_models[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        print(f"Run {run_num}")
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = dict()
        for split_num, per_length_data in per_allele_per_kfold_per_length_binders_train[allele_name].items():
            model = create_model_with_separated_amino_acids(TARGET_LENGTH)
            model.name = model.name + f"run-{run_num}_split-{split_num}"
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = create_model_with_separated_amino_acids(TARGET_LENGTH)


Run 0
Run 1


In [6]:
with open(f"state_graph_cropped.png", 'w+') as f:
    model.plot(file=f, crop_zero=True)

## 2.2 Training process

In [7]:
cutoff = 0.00005
train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][9])
## Stop threshold Hardcoded here
stop_threshold =  abs(cutoff * train_size * -22)
stop_threshold

26.739900000000002

In [8]:
per_allele_per_run_per_split_new_models = dict()
per_allele_per_run_per_split_new_histories = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_new_models[allele_name] = dict()
    per_allele_per_run_per_split_new_histories[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_kfold_per_length_data =  per_allele_per_kfold_per_length_binders_train[allele_name]
        models, histories = train_model(allele_name,
                                        per_kfold_per_length_data,
                                        TARGET_LENGTHS,
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test[allele_name],
                                        multiple_check_input=False,
                                        verbose=True,
                                        prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num],
                                        stop_threshold=stop_threshold,
                                        min_iterations=5)
        per_allele_per_run_per_split_new_models[allele_name][run_num] = models
        per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories



split 0
train size 24309
Shuffling for fairness 
Simple training: Train Length 24309, init_batch_size None
[1] Improvement: 91402.47904651007	Time (s): 3.954
[2] Improvement: 0.0	Time (s): 3.984
[3] Improvement: 0.0	Time (s): 3.989
[4] Improvement: 0.0	Time (s): 3.979
[5] Improvement: 0.0	Time (s): 3.898
Total Training Improvement: 91402.47904651007
Total Training Time (s): 24.2090
split 0
train size 24309
Shuffling for fairness 
Simple training: Train Length 24309, init_batch_size None
[1] Improvement: 90137.96083974431	Time (s): 4.144
[2] Improvement: 1.1641532182693481e-10	Time (s): 4.068
[3] Improvement: 0.0	Time (s): 3.94
[4] Improvement: 0.0	Time (s): 4.005
[5] Improvement: 0.0	Time (s): 3.899
Total Training Improvement: 90137.96083974442
Total Training Time (s): 24.2890


In [9]:
for allele_name, per_run_per_split_new_models in per_allele_per_run_per_split_new_models.items():
    for run_num, per_split_new_models  in per_run_per_split_new_models.items():
        for split_num, model in per_split_new_models.items():
            model.name = model.name + f"run-{run_num}_split-{split_num}"
            history = per_allele_per_run_per_split_new_histories[allele_name][run_num][split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)

In [10]:
import importlib
import hmm_visualization_methods #import the module here, so that it can be reloaded.
importlib.reload(hmm_visualization_methods)

<module 'hmm_visualization_methods' from 'D:\\Projects\\MHC_predictor\\experiments\\length_dependent_hmm\\hmm_visualization_methods.py'>

In [11]:
from hmm_visualization_methods import *

In [12]:
per_name_model = dict()
for allele_name, per_run_per_split_new_models in per_allele_per_run_per_split_new_models.items():
    for run_num, per_split_new_models  in per_run_per_split_new_models.items():
        for split_num, model in per_split_new_models.items():
            per_name_model[model.name] = model

            # print(model_name)

In [13]:
%%time

for i, model in enumerate(per_name_model.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
Wall time: 1.31 s


In [14]:
%%time

for i, model in enumerate(per_name_model.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
1
Wall time: 32.9 s


In [15]:
per_name_pyviz = dict()
for name, model in per_name_model.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (splited_modelrun-0_split-0) (Delete zero edges)
Converting graph for model (splited_modelrun-1_split-0) (Delete zero edges)


In [24]:
model.in_transitions()

AttributeError: 'pomegranate.hmm.HiddenMarkovModel' object has no attribute 'in_transitions'